# Calibration Cleaning

Label-Trim audits the highest-score calibration candidates and removes only those labeled as anomalies before computing conformal p-values.

In [1]:
import numpy as np
import pandas as pd
from oddball import Dataset, load
from pyod.models.hbos import HBOS
from scipy.stats import false_discovery_control

from nonconform.cleaning import apply_label_trim, select_label_trim_candidates
from nonconform.metrics import false_discovery_rate, statistical_power
from nonconform.scoring import Empirical

In [2]:
x_train, x_test, y_test = load(Dataset.SHUTTLE, setup=True, seed=1)
x_train = np.asarray(x_train)
x_test = np.asarray(x_test)
y_test = np.asarray(y_test)

split_idx = int(0.8 * len(x_train))
x_fit = x_train[:split_idx]
x_calib_clean = x_train[split_idx:]

n_injected = 75
anomaly_idx = np.flatnonzero(y_test == 1)
injected_idx = anomaly_idx[:n_injected]
eval_mask = np.ones(len(x_test), dtype=bool)
eval_mask[injected_idx] = False

x_calib = np.vstack([x_calib_clean, x_test[injected_idx]])
y_calib = np.concatenate(
    [np.zeros(len(x_calib_clean), dtype=int), y_test[injected_idx].astype(int)]
)
x_eval = x_test[eval_mask]
y_eval = y_test[eval_mask]

base_detector = HBOS(n_bins=30)
base_detector.fit(x_fit)

calibration_scores = base_detector.decision_function(x_calib)
test_scores = base_detector.decision_function(x_eval)

label_budget = 150
candidate_indices = select_label_trim_candidates(
    calibration_scores,
    label_budget=label_budget,
)
candidate_labels = y_calib[candidate_indices]
trim = apply_label_trim(calibration_scores, candidate_indices, candidate_labels)

print(f"Calibration samples: {len(x_calib)} ({int(y_calib.sum())} injected anomalies)")
print(f"Candidates audited: {trim.n_candidates}; anomalies removed: {trim.n_removed}")
print(f"Evaluation samples: {len(x_eval)} ({int(y_eval.sum())} anomalies)")

Calibration samples: 4634 (75 injected anomalies)
Candidates audited: 150; anomalies removed: 70
Evaluation samples: 925 (25 anomalies)


In [3]:
alpha = 0.15
estimation = Empirical()

standard_p = estimation.compute_p_values(test_scores, calibration_scores)
cleaned_p = estimation.compute_p_values(test_scores, trim.trimmed_scores)
rows = []
for name, p_values in [
    ("before cleaning", standard_p),
    ("after cleaning", cleaned_p),
]:
    discoveries = false_discovery_control(p_values, method="bh") <= alpha
    rows.append(
        {
            "method": name,
            "discoveries": int(discoveries.sum()),
            "fdr": false_discovery_rate(y_eval, discoveries),
            "power": statistical_power(y_eval, discoveries),
        }
    )

results = pd.DataFrame(rows)
print(results.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

         method  discoveries    fdr  power
before cleaning            0 0.0000 0.0000
 after cleaning           27 0.1111 0.9600
